In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

LOGS = Path("../data/captures")

scan = pd.read_json(
    LOGS / "zeek_portscan_01" / "conn.log",
    lines=True
)

benign = pd.read_json(
    LOGS / "zeek_benign_01" / "conn.log",
    lines=True
)

CIC_FILE = Path(
    "../data/MachineLearningCSV/MachineLearningCVE/"
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv"
)

cic = pd.read_csv(CIC_FILE)
cic.columns = cic.columns.str.strip()

print("Zeek scan:", scan.shape)
print("Zeek benign:", benign.shape)
print("CIC dataset:", cic.shape)
print(cic["Label"].value_counts())

Zeek scan: (65554, 21)
Zeek benign: (3, 21)
CIC dataset: (286467, 79)
Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64


In [3]:
def zeek_to_common(conn: pd.DataFrame) -> pd.DataFrame:
    d = conn.copy()

    numeric_cols = [
        "duration", "orig_bytes", "resp_bytes",
        "orig_pkts", "resp_pkts", "id.resp_p"
    ]

    for col in numeric_cols:
        d[col] = pd.to_numeric(d[col], errors="coerce")

    d[numeric_cols] = d[numeric_cols].fillna(0)

    hist = d["history"].fillna("").astype(str)
    total_s = d["duration"].replace(0, np.nan)

    out = pd.DataFrame({
        "dst_port": d["id.resp_p"],
        "duration_us": d["duration"] * 1_000_000,
        "fwd_pkts": d["orig_pkts"],
        "bwd_pkts": d["resp_pkts"],
        "fwd_bytes": d["orig_bytes"],
        "bwd_bytes": d["resp_bytes"],
        "bytes_per_s": (
            (d["orig_bytes"] + d["resp_bytes"]) / total_s
        ).fillna(0),
        "pkts_per_s": (
            (d["orig_pkts"] + d["resp_pkts"]) / total_s
        ).fillna(0),
        "syn_count": hist.str.count("[SsHh]"),
        "rst_count": hist.str.count("[Rr]"),
        "fin_count": hist.str.count("[Ff]"),
    })

    return out

In [4]:
CIC_RENAME = {
    "Destination Port": "dst_port",
    "Flow Duration": "duration_us",
    "Total Fwd Packets": "fwd_pkts",
    "Total Backward Packets": "bwd_pkts",
    "Total Length of Fwd Packets": "fwd_bytes",
    "Total Length of Bwd Packets": "bwd_bytes",
    "Flow Bytes/s": "bytes_per_s",
    "Flow Packets/s": "pkts_per_s",
    "SYN Flag Count": "syn_count",
    "RST Flag Count": "rst_count",
    "FIN Flag Count": "fin_count",
}

def cic_to_common(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=lambda c: c.strip())

    out = df[list(CIC_RENAME)].rename(columns=CIC_RENAME)

    for col in out.columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    return out.replace([np.inf, -np.inf], np.nan)

In [5]:
zeek_scan = zeek_to_common(scan)
zeek_benign = zeek_to_common(benign)

cic_common = cic_to_common(cic)

cic_scan = cic_common.loc[
    cic["Label"].str.strip().eq("PortScan")
].copy()

cic_benign = cic_common.loc[
    cic["Label"].str.strip().eq("BENIGN")
].copy()

print("Zeek scan:", zeek_scan.shape)
print("CIC scan:", cic_scan.shape)
print("Zeek benign:", zeek_benign.shape)
print("CIC benign:", cic_benign.shape)

Zeek scan: (65554, 11)
CIC scan: (158930, 11)
Zeek benign: (3, 11)
CIC benign: (127537, 11)


In [6]:
features = ["fwd_pkts", "duration_us", "syn_count"]

median_comparison = pd.DataFrame({
    "Zeek port scan": zeek_scan[features].median(),
    "CIC port scan": cic_scan[features].median(),
    "Zeek benign": zeek_benign[features].median(),
    "CIC benign": cic_benign[features].median(),
})

median_comparison

,Zeek port scan,CIC port scan,Zeek benign,CIC benign
fwd_pkts,1.000000,1.0,1.700000e+01,2.0
duration_us,28555.512428,47.0,1.718235e+07,30964.0
syn_count,1.000000,0.0,2.000000e+00,0.0
